In [1]:
"""
MLflow Global Model Framework Demo: Food Delivery Time Prediction
================================================================

This demo showcases a "Coffee Machine" approach to ML model deployment:
- Same model architecture deployed globally
- Local fine-tuning for market-specific patterns
- Centralized model registry and management
- Clear performance comparisons between global vs local models

Business Case: Demonstrate measurable improvements when city-specific 
models are used instead of a one-size-fits-all global approach.
"""

import pandas as pd
import numpy as np
import os
import joblib
import mlflow
import mlflow.pyfunc
from mlflow.models import infer_signature
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Configure MLflow connection
mlflow.set_tracking_uri("http://127.0.0.1:8080")
print("MLflow tracking URI set to: http://127.0.0.1:8080")

MLflow tracking URI set to: http://127.0.0.1:8080


In [2]:
# ============================================================================
# STEP 1: GENERATE REALISTIC MULTI-MARKET DATA
# ============================================================================

def generate_delivery_data(n_samples=8000, city="Global", seed=42):
    """
    Generate synthetic food delivery data with city-specific patterns.
    
    This function creates realistic delivery data where each city has distinct
    characteristics (NYC = urban density, Phoenix = suburban efficiency, etc.)
    
    Key Design Decision: Each city has pronounced differences to ensure
    city-specific models can learn meaningful local patterns.
    
    Args:
        n_samples: Number of delivery records to generate
        city: Target city (affects delivery patterns)
        seed: Random seed for reproducibility
    
    Returns:
        DataFrame with delivery features and target delivery times
    """
    np.random.seed(seed)
    data = {}
    
    # === UNIVERSAL FEATURES (same patterns across all cities) ===
    
    # Distance: Exponential distribution (most deliveries are short distance)
    data['distance_km'] = np.clip(np.random.exponential(2.5, n_samples), 0.5, 15.0)
    
    # Restaurant prep time: Normal distribution around 15 minutes
    data['restaurant_prep_time'] = np.clip(np.random.normal(15, 8, n_samples), 3, 45)
    
    # Driver availability: Poisson distribution (realistic staffing patterns)
    data['drivers_available'] = np.clip(np.random.poisson(12, n_samples), 1, 30)
    
    # Time features
    data['hour_of_day'] = np.random.randint(8, 23, n_samples)  # Business hours
    data['day_of_week'] = np.random.randint(0, 7, n_samples)   # 0=Monday, 6=Sunday
    
    # Weather: Realistic distribution (clear weather most common)
    weather_conditions = ['clear', 'light_rain', 'heavy_rain', 'snow', 'fog']
    weather_weights = [0.6, 0.2, 0.1, 0.05, 0.05]
    data['weather_condition'] = np.random.choice(weather_conditions, n_samples, p=weather_weights)
    
    # Restaurant types and order characteristics
    restaurant_types = ['fast_food', 'casual', 'upscale', 'coffee', 'dessert']
    data['restaurant_type'] = np.random.choice(restaurant_types, n_samples)
    data['order_value'] = np.clip(np.random.lognormal(3, 0.8, n_samples), 10, 200)
    data['num_items'] = np.clip(np.random.poisson(2.5, n_samples), 1, 10)
    
    # === BASE DELIVERY TIME CALCULATION ===
    
    # Core delivery time formula (same logic everywhere)
    base_time = (
        data['distance_km'] * 2.5 +           # ~2.5 minutes per km travel
        data['restaurant_prep_time'] +        # Restaurant preparation time
        np.random.normal(5, 2, n_samples) +   # Base delivery overhead
        (data['num_items'] - 1) * 0.5         # Extra complexity per item
    )
    
    # === UNIVERSAL MODIFIERS (apply to all cities) ===
    
    # Weather impact on delivery speed
    weather_multiplier = {'clear': 1.0, 'light_rain': 1.15, 'heavy_rain': 1.35, 'snow': 1.5, 'fog': 1.2}
    for i, weather in enumerate(data['weather_condition']):
        base_time[i] *= weather_multiplier[weather]
    
    # Driver scarcity impact (fewer drivers = longer wait times)
    driver_wait = np.maximum(0, (8 - data['drivers_available']) * 1.2)
    base_time += driver_wait
    
    # Rush hour impact (lunch and dinner time slowdowns)
    rush_multiplier = np.ones(n_samples)
    for i, hour in enumerate(data['hour_of_day']):
        if hour in [12, 13, 18, 19, 20]:  # Peak hours
            rush_multiplier[i] = 1.25
    base_time *= rush_multiplier
    
    # Weekend efficiency (slightly faster due to less traffic)
    weekend_multiplier = np.where(data['day_of_week'] >= 5, 0.95, 1.0)
    base_time *= weekend_multiplier
    
    # === CITY-SPECIFIC PATTERNS (This is where local expertise matters!) ===
    
    if city == "NYC":
        print(f"   Generating NYC data with extreme urban complexity...")
        # NYC challenges: Dense urban environment, parking issues, subway delays
        base_time *= 1.8  # 80% slower due to urban density
        base_time += np.random.choice([0, 15], n_samples, p=[0.6, 0.4])  # Manhattan gridlock
        base_time += np.random.choice([0, 12], n_samples, p=[0.8, 0.2])  # Subway delays affect traffic
        base_time += np.clip(np.random.normal(8, 4, n_samples), 0, 20)   # Parking chaos variability
        
    elif city == "Phoenix":
        print(f"   Generating Phoenix data with desert suburban patterns...")
        # Phoenix advantages: Suburban efficiency, grid layout
        # Phoenix challenges: Extreme heat affects drivers
        base_time *= 0.7  # 30% faster due to suburban roads and layout
        base_time += np.random.choice([0, 20], n_samples, p=[0.6, 0.4])  # Heat impact on drivers
        base_time += np.random.choice([0, 35], n_samples, p=[0.95, 0.05]) # Rare dust storms
        base_time -= np.random.normal(5, 2, n_samples)  # Generally more efficient
        
    elif city == "London":
        print(f"   Generating London data with British complexity...")
        # London challenges: Complex street layout, frequent rain, tube strikes
        base_time *= 1.5  # 50% slower due to complex historical street layout
        base_time += np.random.choice([0, 25], n_samples, p=[0.85, 0.15]) # Occasional tube strikes
        base_time += np.where(np.isin(data['weather_condition'], ['light_rain', 'heavy_rain']),
                             np.random.normal(12, 3, n_samples), 0)        # Rain is very common, affects delivery
        base_time += np.random.choice([0, 18], n_samples, p=[0.9, 0.1])   # Football match days
        
    elif city == "Tokyo":
        print(f"   Generating Tokyo data with precision vs chaos...")
        # Tokyo: Highly efficient systems vs extreme density during rush
        base_time *= 0.8  # 20% faster due to Japanese operational efficiency
        base_time += np.random.choice([0, 45], n_samples, p=[0.92, 0.08]) # Rare natural disasters
        base_time += np.where(np.isin(data['hour_of_day'], [7, 8, 17, 18, 19]),
                             np.random.normal(20, 5, n_samples), 0)        # Extreme rush hour impact
        base_time -= np.random.normal(3, 1, n_samples)  # General precision efficiency bonus
        
    elif city == "Mumbai":
        print(f"   Generating Mumbai data with monsoon chaos...")
        # Mumbai challenges: Most complex traffic, monsoon flooding, high variability
        base_time *= 2.2  # 120% slower - most challenging delivery environment
        base_time += np.random.choice([0, 40], n_samples, p=[0.7, 0.3])   # Monsoon flooding impact
        base_time += np.random.choice([0, 25], n_samples, p=[0.85, 0.15]) # Festival/celebration disruptions
        base_time += np.random.choice([0, 15], n_samples, p=[0.75, 0.25]) # Train delays affect road traffic
    
    # === FINAL PROCESSING ===
    
    # Add realistic noise to prevent overfitting
    base_time += np.random.normal(0, 3, n_samples)
    
    # Ensure realistic delivery time range (8-120 minutes)
    data['delivery_time_minutes'] = np.clip(base_time, 8, 120)
    data['city'] = city
    
    return pd.DataFrame(data)

print("STEP 1: GENERATING MULTI-MARKET DATA")
print("=" * 50)
print("Creating realistic delivery data with pronounced city-specific patterns...")

# === GENERATE TRAINING DATA ===

# Global training data: Mix of all cities to train a generalist model
print("Generating mixed global training data...")
global_data = pd.concat([
    generate_delivery_data(1500, "NYC", seed=42),
    generate_delivery_data(1500, "Phoenix", seed=43), 
    generate_delivery_data(1500, "London", seed=44),
    generate_delivery_data(1500, "Tokyo", seed=45),
    generate_delivery_data(1500, "Mumbai", seed=46),
    generate_delivery_data(2000, "Global", seed=47)  # Generic baseline data
])

# City-specific training data: Pure city data for specialist models
print("Generating city-specific training datasets...")
city_datasets = {
    'NYC': generate_delivery_data(6000, "NYC", seed=50),
    'Phoenix': generate_delivery_data(6000, "Phoenix", seed=51),
    'London': generate_delivery_data(6000, "London", seed=52),
    'Tokyo': generate_delivery_data(6000, "Tokyo", seed=53),
    'Mumbai': generate_delivery_data(6000, "Mumbai", seed=54)
}

# === DATA SUMMARY ===
print(f"\nGenerated {len(global_data):,} global training samples")
for city, data in city_datasets.items():
    avg_time = data['delivery_time_minutes'].mean()
    print(f"{city}: {len(data):,} samples, avg={avg_time:.1f}min")

print("\nMarket Differences (Average Delivery Time):")
print("These differences are what city-specific models should learn to handle:")
for city, data in city_datasets.items():
    avg_time = data['delivery_time_minutes'].mean()
    print(f"   {city:10}: {avg_time:5.1f} minutes")

print("\n" + "=" * 80 + "\n")

STEP 1: GENERATING MULTI-MARKET DATA
Creating realistic delivery data with pronounced city-specific patterns...
Generating mixed global training data...
   Generating NYC data with extreme urban complexity...
   Generating Phoenix data with desert suburban patterns...
   Generating London data with British complexity...
   Generating Tokyo data with precision vs chaos...
   Generating Mumbai data with monsoon chaos...
Generating city-specific training datasets...
   Generating NYC data with extreme urban complexity...
   Generating Phoenix data with desert suburban patterns...
   Generating London data with British complexity...
   Generating Tokyo data with precision vs chaos...
   Generating Mumbai data with monsoon chaos...

Generated 9,500 global training samples
NYC: 6,000 samples, avg=74.1min
Phoenix: 6,000 samples, avg=28.0min
London: 6,000 samples, avg=57.2min
Tokyo: 6,000 samples, avg=32.1min
Mumbai: 6,000 samples, avg=85.3min

Market Differences (Average Delivery Time):
These

In [3]:
# ============================================================================
# STEP 2: DEFINE GLOBALMODEL FRAMEWORK
# ============================================================================

class DeliveryTimeGlobalModel(mlflow.pyfunc.PythonModel):
    """
    GlobalModel Framework for Food Delivery Time Prediction
    
    The 'Coffee Machine' Approach:
    - Same model architecture used globally
    - Consistent preprocessing pipeline  
    - Multi-strategy prediction system
    - Local fine-tuning capabilities
    
    This design ensures operational consistency while allowing
    for market-specific optimization.
    """
    
    def __init__(self):
        self.preprocessor = None
        self.fitted_models = {}  # Dictionary of trained ML models
        self.strategies = pd.DataFrame(columns=['strategy', 'model_list', 'weights'])
        self.initialize_models()
    
    def initialize_models(self):
        """
        Define the ensemble of ML models with hyperparameter grids.
        
        Design Decision: Use multiple algorithms to capture different
        patterns in the data (linear, tree-based, boosting).
        """
        self.models = {
            # Random Forest: Good for capturing feature interactions
            'random_forest': (RandomForestRegressor(random_state=42), {
                'n_estimators': [50, 100], 'max_depth': [None, 10]
            }),
            # XGBoost: Excellent for complex non-linear patterns
            'xgboost': (XGBRegressor(random_state=42), {
                'n_estimators': [50, 100], 'max_depth': [3, 6]
            }),
            # Gradient Boosting: Good baseline boosting algorithm
            'gradient_boosting': (GradientBoostingRegressor(random_state=42), {
                'n_estimators': [50, 100], 'max_depth': [3, 5]
            })
        }
    
    def feature_engineering(self, X):
        """
        Create consistent preprocessing pipeline across all markets.
        
        Critical Design Decision: Same preprocessing everywhere ensures
        models can be deployed consistently but still learn local patterns
        through the training data.
        """
        X = X.copy()
        
        # Separate feature types for different preprocessing
        categorical_features = ['weather_condition', 'restaurant_type', 'city']
        numerical_features = [col for col in X.columns if col not in categorical_features]
        
        # Numerical features: Impute missing values, then standardize
        numeric_transformer = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        
        # Categorical features: Impute missing values, then one-hot encode
        categorical_transformer = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ])
        
        # Combined preprocessing pipeline
        preprocessor = ColumnTransformer([
            ('num', numeric_transformer, numerical_features),
            ('cat', categorical_transformer, categorical_features)
        ])
        
        X_processed = preprocessor.fit_transform(X)
        self.preprocessor = preprocessor
        return X_processed
    
    def fit(self, X_train_processed, y_train, save_path="models"):
        """
        Train all models with hyperparameter optimization.
        
        Each model is tuned via grid search to find optimal parameters,
        then saved for later ensemble use.
        """
        os.makedirs(save_path, exist_ok=True)
        
        for model_name, (model, param_grid) in self.models.items():
            print(f"   Training {model_name}...")
            
            # Hyperparameter tuning via cross-validation
            grid_search = GridSearchCV(
                model, param_grid, cv=3, n_jobs=-1, 
                scoring='neg_mean_absolute_error'
            )
            grid_search.fit(X_train_processed, y_train)
            
            # Save the best model
            best_model = grid_search.best_estimator_
            self.fitted_models[model_name] = best_model
            joblib.dump(best_model, os.path.join(save_path, f"{model_name}.pkl"))
    
    def add_strategies(self, strategies_df):
        """
        Add prediction strategies for different scenarios.
        
        Strategy Examples:
        - 'balanced': Equal weighting of all models
        - 'rush_hour': Emphasize models that handle time-based patterns
        - 'weather_sensitive': Focus on weather-aware models
        """
        self.strategies = pd.concat([self.strategies, strategies_df], ignore_index=True)
    
    def predict(self, context, model_input):
        """
        Make predictions using ensemble strategy.
        
        The strategy system allows different model weightings
        for different scenarios (e.g., rush hour vs normal conditions).
        """
        # Extract strategy if provided
        if 'strategy' in model_input.columns:
            strategy = model_input['strategy'].iloc[0]
            model_input = model_input.drop(columns=['strategy'])
        else:
            strategy = 'balanced'
        
        # Find the specified strategy
        strategy_row = self.strategies[self.strategies['strategy'] == strategy]
        if strategy_row.empty:
            # Default to balanced ensemble
            model_list = list(self.fitted_models.keys())
            weights = [1.0 / len(model_list)] * len(model_list)
        else:
            model_list = strategy_row['model_list'].iloc[0].split(',')
            weights = [float(w) for w in strategy_row['weights'].iloc[0].split(',')]
            weights = np.array(weights) / np.sum(weights)  # Normalize weights
        
        # Preprocess features
        if self.preprocessor is None:
            X_processed = self.feature_engineering(model_input)
        else:
            X_processed = self.preprocessor.transform(model_input)
        
        # Get predictions from each model in the ensemble
        predictions = []
        for model_name in model_list:
            if model_name in self.fitted_models:
                pred = self.fitted_models[model_name].predict(X_processed)
                predictions.append(pred)
        
        # Weighted average of predictions
        if len(predictions) > 1:
            predictions = np.array(predictions)
            return np.average(predictions, axis=0, weights=weights)
        else:
            return predictions[0] if predictions else np.zeros(len(model_input))
    
    def load_context(self, context):
        """Load model artifacts when deployed via MLflow"""
        # Load preprocessor
        preprocessor_path = context.artifacts.get("preprocessor")
        if preprocessor_path:
            self.preprocessor = joblib.load(preprocessor_path)
        
        # Load individual models
        for model_name in self.models.keys():
            model_path = context.artifacts.get(model_name)
            if model_path:
                self.fitted_models[model_name] = joblib.load(model_path)

def fix_data_types(df):
    """
    Ensure proper data types for MLflow compatibility.
    
    MLflow requires consistent data types for model signatures.
    This function standardizes all input features.
    """
    df = df.copy()
    
    # Define expected data types
    float_cols = ['distance_km', 'restaurant_prep_time', 'order_value']
    int_cols = ['drivers_available', 'hour_of_day', 'day_of_week', 'num_items']
    str_cols = ['weather_condition', 'restaurant_type', 'city']
    
    # Apply type conversions
    for col in float_cols:
        if col in df.columns:
            df[col] = df[col].astype(float)
    
    for col in int_cols:
        if col in df.columns:
            df[col] = df[col].astype(int)
            
    for col in str_cols:
        if col in df.columns:
            df[col] = df[col].astype(str)
    
    return df

print("STEP 2: GLOBALMODEL FRAMEWORK READY")
print("=" * 40)
print("Coffee Machine architecture defined:")
print("- Consistent preprocessing pipeline")
print("- Multi-model ensemble approach")
print("- Strategy-based prediction system")
print("- MLflow integration for deployment")
print("\n" + "=" * 80 + "\n")

STEP 2: GLOBALMODEL FRAMEWORK READY
Coffee Machine architecture defined:
- Consistent preprocessing pipeline
- Multi-model ensemble approach
- Strategy-based prediction system
- MLflow integration for deployment




/Users/panderah/.pyenv/versions/mlflowAgent/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:168: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [4]:
# ============================================================================
# STEP 3: TRAIN AND REGISTER GLOBAL MODEL
# ============================================================================

print("STEP 3: TRAINING GLOBAL MODEL")
print("=" * 35)
print("Training a generalist model on mixed data from all cities...")
print("This model will serve as our baseline and starting point for local fine-tuning.")

# === PREPARE GLOBAL TRAINING DATA ===
X_global = global_data.drop(['delivery_time_minutes'], axis=1)
y_global = global_data['delivery_time_minutes']

# Split for training and testing
X_train_global, X_test_global, y_train_global, y_test_global = train_test_split(
    X_global, y_global, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train_global):,} samples")
print(f"Test set: {len(X_test_global):,} samples")

# === TRAIN GLOBAL MODEL ===
print("Initializing and training global model...")
global_model = DeliveryTimeGlobalModel()

# Feature engineering and preprocessing
print("   Applying feature engineering...")
X_train_processed_global = global_model.feature_engineering(X_train_global)

print("   Training ensemble of models...")
global_model.fit(X_train_processed_global, y_train_global, "delivery_models")

# === ADD PREDICTION STRATEGIES ===
print("   Adding prediction strategies...")
global_strategies = pd.DataFrame({
    'strategy': ['balanced', 'rush_hour', 'weather_sensitive'],
    'model_list': [
        'random_forest,xgboost,gradient_boosting',  # All models equally weighted
        'xgboost,gradient_boosting',                # Focus on boosting for temporal patterns
        'random_forest,xgboost,gradient_boosting'   # All models with weather emphasis
    ],
    'weights': ['0.4,0.4,0.2', '0.7,0.3', '0.3,0.4,0.3']
})
global_model.add_strategies(global_strategies)

# Save preprocessor
joblib.dump(global_model.preprocessor, 'delivery_models/preprocessor.pkl')

# === CREATE DEPLOYMENT ENVIRONMENT ===
conda_env_path = "delivery_models/conda_env.yaml"
with open(conda_env_path, 'w') as f:
    f.write("""
name: delivery_env
channels:
  - conda-forge
dependencies:
  - python=3.8
  - scikit-learn=1.3.0
  - xgboost=2.0.3
  - joblib=1.2.0
  - pandas=1.5.3
  - numpy=1.23.5
  - pip:
    - mlflow==2.21.0
""")

# === REGISTER GLOBAL MODEL IN MLFLOW ===
print("   Registering global model in MLflow Model Registry...")

mlflow.set_experiment("DeliveryTime_GlobalModel")

# Create model signature for MLflow
example_input = X_train_global[:1].copy()
example_input['strategy'] = 'balanced'
example_input = fix_data_types(example_input)
example_output = y_train_global[:1]
signature = infer_signature(example_input, example_output)

# Define model artifacts
artifacts = {
    model_name: os.path.join("delivery_models", f"{model_name}.pkl") 
    for model_name in global_model.models.keys()
}
artifacts["preprocessor"] = "delivery_models/preprocessor.pkl"

# Log and register the model
with mlflow.start_run(run_name="Global_Model") as run:
    mlflow.log_param("model_type", "GlobalModel")
    mlflow.log_param("training_samples", len(X_train_global))
    
    # Log the model with all artifacts
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=global_model,
        artifacts=artifacts,
        conda_env=conda_env_path,
        signature=signature
    )
    
    # Register in Model Registry for production use
    model_uri = f"runs:/{run.info.run_id}/model"
    mlflow.register_model(model_uri, "delivery-time-global")

print("Global model successfully registered as 'delivery-time-global'")
print("\n" + "=" * 80 + "\n")

STEP 3: TRAINING GLOBAL MODEL
Training a generalist model on mixed data from all cities...
This model will serve as our baseline and starting point for local fine-tuning.
Training set: 7,600 samples
Test set: 1,900 samples
Initializing and training global model...
   Applying feature engineering...
   Training ensemble of models...
   Training random_forest...
   Training xgboost...
   Training gradient_boosting...
   Adding prediction strategies...
   Registering global model in MLflow Model Registry...


Registered model 'delivery-time-global' already exists. Creating a new version of this model...
2025/08/28 08:44:25 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: delivery-time-global, version 6


🏃 View run Global_Model at: http://127.0.0.1:8080/#/experiments/107297747479459249/runs/3c492dfdb3b84c13963743669f6a9078
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/107297747479459249
Global model successfully registered as 'delivery-time-global'




Created version '6' of model 'delivery-time-global'.


In [5]:
# ============================================================================
# STEP 4: MODEL VALIDATION
# ============================================================================

def validate_model(model_name, model_version, test_data_sample, expected_output_range=(5, 120)):
    """
    Validate that a registered MLflow model works correctly.
    
    This is a critical step before deploying any model to production.
    We test basic functionality and output sanity.
    
    Returns:
        tuple: (is_valid, validation_message, sample_prediction)
    """
    try:
        # Load model from MLflow registry
        model_uri = f"models:/{model_name}/{model_version}"
        model = mlflow.pyfunc.load_model(model_uri)
        
        # Test prediction
        sample_pred = model.predict(test_data_sample)
        
        # Validate prediction range
        if len(sample_pred) > 0:
            pred_value = sample_pred[0]
            if expected_output_range[0] <= pred_value <= expected_output_range[1]:
                return True, f"Model validation passed", pred_value
            else:
                return False, f"Prediction {pred_value:.1f} outside expected range", pred_value
        else:
            return False, "No prediction returned", None
            
    except Exception as e:
        return False, f"Validation failed: {str(e)[:50]}...", None

print("STEP 4: VALIDATING GLOBAL MODEL")
print("=" * 38)
print("Testing global model with a sample prediction...")

# Create a realistic validation sample
validation_sample = pd.DataFrame([{
    'distance_km': 2.5,
    'restaurant_prep_time': 15.0,
    'drivers_available': 8,
    'hour_of_day': 18,
    'day_of_week': 2,
    'weather_condition': 'clear',
    'restaurant_type': 'casual',
    'order_value': 35.0,
    'num_items': 2,
    'city': 'NYC',
    'strategy': 'balanced'
}])

validation_sample = fix_data_types(validation_sample)

# Run validation
is_valid, message, pred = validate_model("delivery-time-global", "latest", validation_sample)
print(f"Global model validation: {message}")
if pred is not None:
    print(f"Sample prediction: {pred:.1f} minutes")

print("\n" + "=" * 80 + "\n")

STEP 4: VALIDATING GLOBAL MODEL
Testing global model with a sample prediction...


Global model validation: Model validation passed
Sample prediction: 70.8 minutes




In [6]:
# ============================================================================
# STEP 5: FINE-TUNE AND REGISTER CITY MODELS
# ============================================================================

print("STEP 5: FINE-TUNING CITY-SPECIFIC MODELS")
print("=" * 48)
print("Creating specialized models for each market using local data...")
print("Each city model starts from the global model and fine-tunes on local patterns.")

# === DEFINE CITY-SPECIFIC STRATEGIES ===
print("Defining city-specific prediction strategies...")
city_strategies = {
    # NYC strategies emphasize weather and urban complexity handling
    'NYC': pd.DataFrame({
        'strategy': ['balanced', 'manhattan_rush', 'bad_weather'],
        'model_list': ['random_forest,xgboost,gradient_boosting', 'xgboost,gradient_boosting', 'random_forest,xgboost'],
        'weights': ['0.4,0.4,0.2', '0.8,0.2', '0.6,0.4']
    }),
    # Phoenix strategies focus on heat impact and suburban efficiency
    'Phoenix': pd.DataFrame({
        'strategy': ['balanced', 'heat_wave', 'suburban_fast'],
        'model_list': ['random_forest,gradient_boosting', 'xgboost,gradient_boosting', 'random_forest'],
        'weights': ['0.6,0.4', '0.7,0.3', '1.0']
    }),
    # London strategies handle rain and transport disruptions
    'London': pd.DataFrame({
        'strategy': ['balanced', 'tube_strikes', 'rainy_day'],
        'model_list': ['random_forest,xgboost,gradient_boosting', 'xgboost,gradient_boosting', 'random_forest,xgboost'],
        'weights': ['0.4,0.4,0.2', '0.6,0.4', '0.5,0.5']
    }),
    # Tokyo strategies emphasize precision and rush hour handling
    'Tokyo': pd.DataFrame({
        'strategy': ['balanced', 'rush_hour', 'precision_mode'],
        'model_list': ['random_forest,xgboost', 'xgboost,gradient_boosting', 'random_forest'],
        'weights': ['0.6,0.4', '0.8,0.2', '1.0']
    }),
    # Mumbai strategies handle high variability and monsoon impact
    'Mumbai': pd.DataFrame({
        'strategy': ['balanced', 'monsoon', 'festival_day'],
        'model_list': ['xgboost,gradient_boosting', 'xgboost,gradient_boosting', 'gradient_boosting'],
        'weights': ['0.6,0.4', '0.7,0.3', '1.0']
    })
}

successfully_registered = []

# === TRAIN AND REGISTER EACH CITY MODEL ===
for city_name, city_data in city_datasets.items():
    print(f"\nFine-tuning {city_name} model...")
    
    try:
        # Load global model as starting point
        print(f"   Loading global model as base...")
        global_model_uri = "models:/delivery-time-global/latest"
        base_model = mlflow.pyfunc.load_model(global_model_uri)._model_impl.python_model
        
        # Prepare city-specific training data
        print(f"   Preparing {city_name} training data...")
        X_city = city_data.drop(['delivery_time_minutes'], axis=1)
        y_city = city_data['delivery_time_minutes']
        
        X_train_city, X_test_city, y_train_city, y_test_city = train_test_split(
            X_city, y_city, test_size=0.3, random_state=42
        )
        
        # Use global preprocessor for consistency (critical for deployment)
        print(f"   Applying consistent preprocessing...")
        X_train_processed = base_model.preprocessor.transform(X_train_city)
        
        # Fine-tune models on city-specific data
        print(f"   Fine-tuning ensemble on {city_name} data...")
        base_model.fit(X_train_processed, y_train_city, "delivery_models")
        
        # Add city-specific strategies
        print(f"   Adding {city_name}-specific strategies...")
        base_model.add_strategies(city_strategies[city_name])
        
        # Validate before registering
        city_validation_sample = validation_sample.copy()
        city_validation_sample['city'] = city_name
        
        # Register city model in MLflow
        print(f"   Registering {city_name} model...")
        mlflow.set_experiment(f"DeliveryTime_{city_name}")
        with mlflow.start_run(run_name=f"{city_name}_Model") as run:
            mlflow.log_param("model_type", f"GlobalModel_{city_name}")
            mlflow.log_param("base_model", "delivery-time-global")
            mlflow.log_param("training_samples", len(X_train_city))
            
            mlflow.pyfunc.log_model(
                artifact_path="model",
                python_model=base_model,
                artifacts=artifacts,
                conda_env=conda_env_path,
                signature=signature
            )
            
            # Test the model before final registration
            temp_model_uri = f"runs:/{run.info.run_id}/model"
            temp_model = mlflow.pyfunc.load_model(temp_model_uri)
            
            try:
                test_pred = temp_model.predict(city_validation_sample)
                if len(test_pred) > 0 and 5 <= test_pred[0] <= 120:
                    # Model works, register it
                    model_name = f"delivery-time-{city_name.lower()}"
                    mlflow.register_model(temp_model_uri, model_name)
                    successfully_registered.append(city_name)
                    print(f"   {city_name} model registered as '{model_name}'")
                    print(f"   Validation prediction: {test_pred[0]:.1f} minutes")
                else:
                    print(f"   {city_name} model failed validation - prediction out of range")
            except Exception as e:
                print(f"   {city_name} model failed validation: {str(e)[:50]}...")
    
    except Exception as e:
        print(f"   Failed to fine-tune {city_name}: {str(e)[:50]}...")

print(f"\nSuccessfully registered {len(successfully_registered)}/{len(city_datasets)} city models:")
for city in successfully_registered:
    print(f"   {city}: delivery-time-{city.lower()}")

print("\n" + "=" * 80 + "\n")

STEP 5: FINE-TUNING CITY-SPECIFIC MODELS
Creating specialized models for each market using local data...
Each city model starts from the global model and fine-tunes on local patterns.
Defining city-specific prediction strategies...

Fine-tuning NYC model...
   Loading global model as base...


   Preparing NYC training data...
   Applying consistent preprocessing...
   Fine-tuning ensemble on NYC data...
   Training random_forest...
   Training xgboost...
   Training gradient_boosting...
   Adding NYC-specific strategies...
   Registering NYC model...


Registered model 'delivery-time-nyc' already exists. Creating a new version of this model...
2025/08/28 08:45:58 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: delivery-time-nyc, version 6
Created version '6' of model 'delivery-time-nyc'.


   NYC model registered as 'delivery-time-nyc'
   Validation prediction: 75.7 minutes
🏃 View run NYC_Model at: http://127.0.0.1:8080/#/experiments/314427693869722950/runs/6e2f8656086f4dfe8eec671a52dd01df
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/314427693869722950

Fine-tuning Phoenix model...
   Loading global model as base...


   Preparing Phoenix training data...
   Applying consistent preprocessing...
   Fine-tuning ensemble on Phoenix data...
   Training random_forest...
   Training xgboost...
   Training gradient_boosting...
   Adding Phoenix-specific strategies...
   Registering Phoenix model...


...

Registered model 'delivery-time-tokyo' already exists. Creating a new version of this model...
2025/08/28 08:46:11 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: delivery-time-tokyo, version 6
Created version '6' of model 'delivery-time-tokyo'.


   Tokyo model registered as 'delivery-time-tokyo'
   Validation prediction: 47.5 minutes
🏃 View run Tokyo_Model at: http://127.0.0.1:8080/#/experiments/159677554801086918/runs/ea31fb7fa16a4e4eb6406bed3f6750e6
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/159677554801086918

Fine-tuning Mumbai model...
   Loading global model as base...


   Preparing Mumbai training data...
   Applying consistent preprocessing...
   Fine-tuning ensemble on Mumbai data...
   Training random_forest...
   Training xgboost...
   Training gradient_boosting...
   Adding Mumbai-specific strategies...
   Registering Mumbai model...


Registered model 'delivery-time-mumbai' already exists. Creating a new version of this model...
2025/08/28 08:46:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: delivery-time-mumbai, version 6
Created version '6' of model 'delivery-time-mumbai'.


   Mumbai model registered as 'delivery-time-mumbai'
   Validation prediction: 88.8 minutes
🏃 View run Mumbai_Model at: http://127.0.0.1:8080/#/experiments/255267586844495788/runs/23cb0cedbc52445d95d40b21a001962b
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/255267586844495788

Successfully registered 5/5 city models:
   NYC: delivery-time-nyc
   Phoenix: delivery-time-phoenix
   London: delivery-time-london
   Tokyo: delivery-time-tokyo
   Mumbai: delivery-time-mumbai




In [7]:
# ============================================================================
# STEP 6: COMPREHENSIVE MODEL COMPARISON
# ============================================================================

print("STEP 6: COMPREHENSIVE MODEL COMPARISON")
print("=" * 42)
print("Comparing global vs city-specific model performance...")
print("This is where we prove the business value of local optimization.")

def test_model_performance(model_name, test_data, test_labels, strategy='balanced'):
    """
    Test model performance using Mean Absolute Error (MAE).
    
    MAE measures prediction accuracy:
    - Lower MAE = more accurate predictions
    - MAE of 5.0 means predictions are off by 5 minutes on average
    
    This function tests models on held-out test data to measure
    how well they generalize to unseen delivery scenarios.
    """
    try:
        # Load model from MLflow registry
        model = mlflow.pyfunc.load_model(f"models:/{model_name}/latest")
        
        # Prepare test input with strategy
        test_input = test_data.copy()
        test_input['strategy'] = strategy
        test_input = fix_data_types(test_input)
        
        # Get model predictions
        predictions = model.predict(test_input)
        
        # Calculate accuracy metrics
        mae = mean_absolute_error(test_labels, predictions)  # Average prediction error
        r2 = r2_score(test_labels, predictions)              # Variance explained
        
        return mae, r2, True
    except Exception as e:
        print(f"   Error testing {model_name}: {str(e)[:40]}...")
        return 999, 0, False

# === AGGREGATE PERFORMANCE TESTING ===
print("\nAGGREGATE PERFORMANCE TESTING")
print("-" * 35)
print("Testing models on large held-out test sets from each city...")
print("MAE = Mean Absolute Error (lower is better)")
print("R² = Variance Explained (higher is better)")

comparison_results = {}

for city_name, city_data in city_datasets.items():
    print(f"\n{city_name.upper()} MARKET COMPARISON")
    print("-" * 35)
    
    # Create held-out test set for this city
    X_city = city_data.drop(['delivery_time_minutes'], axis=1)
    y_city = city_data['delivery_time_minutes']
    _, X_test, _, y_test = train_test_split(X_city, y_city, test_size=0.3, random_state=42)
    
    print(f"   Testing on {len(X_test):,} {city_name} delivery samples...")
    
    # Test global model (our baseline)
    global_mae, global_r2, global_success = test_model_performance(
        "delivery-time-global", X_test, y_test, "balanced"
    )
    
    print(f"   Global Model: MAE = {global_mae:.2f} min | R² = {global_r2:.3f}")
    print(f"      (Global model predictions are off by {global_mae:.2f} minutes on average)")
    
    # Test city-specific model (our specialist)
    city_model_name = f"delivery-time-{city_name.lower()}"
    city_mae, city_r2, city_success = 999, 0, False
    
    if city_name in successfully_registered:
        city_mae, city_r2, city_success = test_model_performance(
            city_model_name, X_test, y_test, "balanced"
        )
        print(f"   {city_name} Model: MAE = {city_mae:.2f} min | R² = {city_r2:.3f}")
        print(f"      ({city_name} model predictions are off by {city_mae:.2f} minutes on average)")
    else:
        print(f"   {city_name} Model: Not available")
    
    # Calculate improvement
    if global_success and city_success and global_mae > 0:
        improvement = ((global_mae - city_mae) / global_mae) * 100
        winner = "City" if improvement > 0 else "Global"
        
        if improvement > 0:
            print(f"   IMPROVEMENT: {improvement:.1f}% more accurate ({city_name} model wins!)")
            print(f"      City model reduces prediction error by {global_mae - city_mae:.2f} minutes")
        else:
            print(f"   Global model performs better by {abs(improvement):.1f}%")
    else:
        improvement = 0
        winner = "N/A"
        print("   Cannot compare (testing failed)")
    
    # Store results for summary
    comparison_results[city_name] = {
        'global_mae': global_mae,
        'city_mae': city_mae,
        'improvement': improvement,
        'winner': winner,
        'city_available': city_name in successfully_registered
    }

print("\n" + "=" * 80 + "\n")

STEP 6: COMPREHENSIVE MODEL COMPARISON
Comparing global vs city-specific model performance...
This is where we prove the business value of local optimization.

AGGREGATE PERFORMANCE TESTING
-----------------------------------
Testing models on large held-out test sets from each city...
MAE = Mean Absolute Error (lower is better)
R² = Variance Explained (higher is better)

NYC MARKET COMPARISON
-----------------------------------
   Testing on 1,800 NYC delivery samples...


   Global Model: MAE = 9.65 min | R² = 0.745
      (Global model predictions are off by 9.65 minutes on average)


   NYC Model: MAE = 9.30 min | R² = 0.762
      (NYC model predictions are off by 9.30 minutes on average)
   IMPROVEMENT: 3.6% more accurate (NYC model wins!)
      City model reduces prediction error by 0.35 minutes

PHOENIX MARKET COMPARISON
-----------------------------------
   Testing on 1,800 Phoenix delivery samples...


   Global Model: MAE = 10.91 min | R² = 0.287
      (Global model predictions are off by 10.91 minutes on average)


   Phoenix Model: MAE = 10.74 min | R² = 0.304
      (Phoenix model predictions are off by 10.74 minutes on average)
   IMPROVEMENT: 1.6% more accurate (Phoenix model wins!)
      City model reduces prediction error by 0.17 minutes

LONDON MARKET COMPARISON
-----------------------------------
   Testing on 1,800 London delivery samples...


...

   London Model: MAE = 9.48 min | R² = 0.724
      (London model predictions are off by 9.48 minutes on average)
   IMPROVEMENT: 2.7% more accurate (London model wins!)
      City model reduces prediction error by 0.27 minutes

TOKYO MARKET COMPARISON
-----------------------------------
   Testing on 1,800 Tokyo delivery samples...


   Global Model: MAE = 9.19 min | R² = 0.423
      (Global model predictions are off by 9.19 minutes on average)


   Tokyo Model: MAE = 8.49 min | R² = 0.479
      (Tokyo model predictions are off by 8.49 minutes on average)
   IMPROVEMENT: 7.7% more accurate (Tokyo model wins!)
      City model reduces prediction error by 0.70 minutes

MUMBAI MARKET COMPARISON
-----------------------------------
   Testing on 1,800 Mumbai delivery samples...


   Global Model: MAE = 15.92 min | R² = 0.539
      (Global model predictions are off by 15.92 minutes on average)


   Mumbai Model: MAE = 15.68 min | R² = 0.550
      (Mumbai model predictions are off by 15.68 minutes on average)
   IMPROVEMENT: 1.5% more accurate (Mumbai model wins!)
      City model reduces prediction error by 0.24 minutes




In [8]:
# ============================================================================
# STEP 7: SCENARIO-BASED ANALYSIS WITH EXPECTED VALUES
# ============================================================================

print("STEP 7: DETAILED SCENARIO ANALYSIS")
print("=" * 40)
print("Testing specific challenging scenarios where local expertise should matter...")
print("Expected values are calculated based on training data patterns + scenario conditions.")

# === LOAD MODELS FOR SCENARIO TESTING ===
try:
    global_model_loaded = mlflow.pyfunc.load_model("models:/delivery-time-global/latest")
    print("Global model loaded successfully")
except Exception as e:
    print(f"Could not load global model: {e}")
    global_model_loaded = None

city_models_loaded = {}
for city in successfully_registered:
    try:
        model_name = f"delivery-time-{city.lower()}"
        city_models_loaded[city] = mlflow.pyfunc.load_model(f"models:/{model_name}/latest")
        print(f"{city} model loaded successfully")
    except Exception as e:
        print(f"Could not load {city} model: {e}")

print()

# === CALCULATE BASELINE EXPECTATIONS ===
print("Calculating baseline delivery times from training data...")
city_baselines = {}
for city_name, city_data in city_datasets.items():
    city_baselines[city_name] = city_data['delivery_time_minutes'].mean()
    print(f"   {city_name} baseline: {city_baselines[city_name]:.1f} minutes")

print()

# === DEFINE CHALLENGING SCENARIOS ===
print("Creating challenging test scenarios...")

test_scenarios = {
    'NYC_Storm': {
        'name': 'NYC Manhattan Storm Rush',
        'description': 'Heavy rain + rush hour + urban density',
        'data': {
            'distance_km': 5.2, 'restaurant_prep_time': 25.0, 'drivers_available': 3,
            'hour_of_day': 18, 'day_of_week': 2, 'weather_condition': 'heavy_rain',
            'restaurant_type': 'upscale', 'order_value': 75.0, 'num_items': 4, 'city': 'NYC'
        },
        'strategy': 'bad_weather',
        # Expected calculation: baseline + weather penalty + rush hour + low drivers
        'expected': city_baselines['NYC'] * 1.35 + 10  # +35% for storm, +10 min for conditions
    },
    'Phoenix_Heat': {
        'name': 'Phoenix Extreme Heat Wave', 
        'description': 'Afternoon heat affecting driver performance',
        'data': {
            'distance_km': 6.8, 'restaurant_prep_time': 15.0, 'drivers_available': 6,
            'hour_of_day': 15, 'day_of_week': 4, 'weather_condition': 'clear',
            'restaurant_type': 'fast_food', 'order_value': 25.0, 'num_items': 2, 'city': 'Phoenix'
        },
        'strategy': 'heat_wave',
        # Expected calculation: baseline + heat penalty + distance penalty
        'expected': city_baselines['Phoenix'] * 1.2 + 5  # +20% for heat, +5 min for long distance
    },
    'London_Crisis': {
        'name': 'London Tube Strike + Rain',
        'description': 'Transport disruption + weather impact',
        'data': {
            'distance_km': 4.5, 'restaurant_prep_time': 22.0, 'drivers_available': 3,
            'hour_of_day': 19, 'day_of_week': 1, 'weather_condition': 'heavy_rain',
            'restaurant_type': 'casual', 'order_value': 55.0, 'num_items': 4, 'city': 'London'
        },
        'strategy': 'tube_strikes',
        # Expected calculation: baseline + rain + tube strike + rush hour
        'expected': city_baselines['London'] * 1.25 + 15  # +25% for rain, +15 min for disruption
    },
    'Tokyo_Rush': {
        'name': 'Tokyo Extreme Rush Hour',
        'description': 'Peak commuter hour with limited drivers',
        'data': {
            'distance_km': 3.2, 'restaurant_prep_time': 15.0, 'drivers_available': 2,
            'hour_of_day': 18, 'day_of_week': 1, 'weather_condition': 'clear',
            'restaurant_type': 'casual', 'order_value': 45.0, 'num_items': 3, 'city': 'Tokyo'
        },
        'strategy': 'rush_hour',
        # Expected calculation: baseline + extreme rush penalty + driver scarcity
        'expected': city_baselines['Tokyo'] * 1.4 + 8  # +40% for rush, +8 min for low drivers
    },
    'Mumbai_Monsoon': {
        'name': 'Mumbai Monsoon Flooding',
        'description': 'Heavy rain causing street flooding + festival crowds',
        'data': {
            'distance_km': 4.2, 'restaurant_prep_time': 25.0, 'drivers_available': 4,
            'hour_of_day': 20, 'day_of_week': 0, 'weather_condition': 'heavy_rain',
            'restaurant_type': 'casual', 'order_value': 50.0, 'num_items': 4, 'city': 'Mumbai'
        },
        'strategy': 'monsoon',
        # Expected calculation: baseline + monsoon penalty + weekend evening
        'expected': city_baselines['Mumbai'] * 1.5 + 12  # +50% for monsoon, +12 min for flooding
    }
}

# === RUN SCENARIO TESTS ===
print("\nSCENARIO PREDICTIONS vs EXPECTED:")
print("=" * 60)

scenario_results = {}

for scenario_id, scenario in test_scenarios.items():
    city_name = scenario['data']['city']
    expected = scenario['expected']
    
    print(f"\n{scenario['name'].upper()}")
    print(f"Scenario: {scenario['description']}")
    print("-" * 50)
    print(f"Expected Delivery Time: {expected:.1f} minutes")
    print("   (Based on training data + scenario conditions)")
    
    # Create scenario DataFrame
    scenario_df = pd.DataFrame([scenario['data']])
    scenario_df = fix_data_types(scenario_df)
    
    # Get predictions
    global_pred = None
    city_pred = None
    
    # Global model prediction
    if global_model_loaded:
        try:
            global_scenario = scenario_df.copy()
            global_scenario['strategy'] = 'balanced'  # Global uses balanced strategy
            global_pred = global_model_loaded.predict(global_scenario)[0]
        except Exception as e:
            print(f"Global prediction error: {e}")
    
    # City model prediction  
    if city_name in city_models_loaded:
        try:
            city_scenario = scenario_df.copy()
            city_scenario['strategy'] = scenario.get('strategy', 'balanced')
            city_pred = city_models_loaded[city_name].predict(city_scenario)[0]
        except Exception as e:
            print(f"City prediction error: {e}")
    
    # Display results and calculate accuracy
    if global_pred is not None:
        global_error = abs(global_pred - expected)
        print(f"Global Model Prediction: {global_pred:6.1f} minutes (error: {global_error:.1f} min)")
    
    if city_pred is not None:
        city_error = abs(city_pred - expected)
        print(f"{city_name} Model Prediction:  {city_pred:6.1f} minutes (error: {city_error:.1f} min)")
    
    # Determine winner based on accuracy
    if global_pred is not None and city_pred is not None:
        if city_error < global_error:
            improvement = ((global_error - city_error) / global_error) * 100
            winner = f"{city_name} model (more accurate by {improvement:.1f}%)"
            print(f"WINNER: {winner}")
            print(f"   {city_name} model is {global_error - city_error:.1f} minutes closer to expected time")
        elif global_error < city_error:
            improvement = ((city_error - global_error) / city_error) * 100
            winner = f"Global model (more accurate by {improvement:.1f}%)"
            print(f"WINNER: {winner}")
            print(f"   Global model is {city_error - global_error:.1f} minutes closer to expected time")
        else:
            winner = "Tie"
            improvement = 0
            print("WINNER: Tie (equal accuracy)")
        
        # Store results
        scenario_results[scenario_id] = {
            'scenario': scenario['name'],
            'city': city_name,
            'expected': expected,
            'global_pred': global_pred,
            'city_pred': city_pred,
            'global_error': global_error,
            'city_error': city_error,
            'winner': winner,
            'improvement': improvement if city_name in winner else -improvement if 'Global' in winner else 0
        }

print("\n" + "=" * 80 + "\n")

STEP 7: DETAILED SCENARIO ANALYSIS
Testing specific challenging scenarios where local expertise should matter...
Expected values are calculated based on training data patterns + scenario conditions.


Global model loaded successfully


NYC model loaded successfully


Phoenix model loaded successfully


London model loaded successfully


Tokyo model loaded successfully


Mumbai model loaded successfully

Calculating baseline delivery times from training data...
   NYC baseline: 74.1 minutes
   Phoenix baseline: 28.0 minutes
   London baseline: 57.2 minutes
   Tokyo baseline: 32.1 minutes
   Mumbai baseline: 85.3 minutes

Creating challenging test scenarios...

SCENARIO PREDICTIONS vs EXPECTED:

NYC MANHATTAN STORM RUSH
Scenario: Heavy rain + rush hour + urban density
--------------------------------------------------
Expected Delivery Time: 110.0 minutes
   (Based on training data + scenario conditions)
Global Model Prediction:  128.7 minutes (error: 18.7 min)
NYC Model Prediction:   113.6 minutes (error: 3.5 min)
WINNER: NYC model (more accurate by 81.1%)
   NYC model is 15.1 minutes closer to expected time

PHOENIX EXTREME HEAT WAVE
Scenario: Afternoon heat affecting driver performance
--------------------------------------------------
Expected Delivery Time: 38.6 minutes
   (Based on training data + scenario conditions)
Global Model Prediction:   35

In [9]:
# ============================================================================
# STEP 8: COMPREHENSIVE FINAL REPORT
# ============================================================================

print("STEP 8: COMPREHENSIVE BUSINESS CASE REPORT")
print("=" * 48)

print("FOOD DELIVERY TIME PREDICTION - GLOBAL vs CITY-SPECIFIC MODELS")
print("=" * 70)
print("\nEXECUTIVE SUMMARY:")
print("This analysis compares a single global model against city-specific models")
print("to determine if local optimization provides measurable business value.")

# === AGGREGATE PERFORMANCE SUMMARY ===
print(f"\n1. AGGREGATE PERFORMANCE (MAE = Mean Absolute Error)")
print("   Lower MAE = More Accurate Predictions = Better Customer Experience")
print()
print("Market     | Global MAE | Local MAE  | Improvement | Winner")
print("-" * 58)

total_improvements = []
wins = 0
total_comparisons = 0

for city, results in comparison_results.items():
    if results['city_available'] and results['global_mae'] < 900:
        improvement_str = f"{results['improvement']:+6.1f}%"
        winner_symbol = "Local" if results['winner'] == "City" else "Global"
        
        total_improvements.append(results['improvement'])
        total_comparisons += 1
        if results['winner'] == "City":
            wins += 1
            
        print(f"{city:10} | {results['global_mae']:8.2f}   | {results['city_mae']:8.2f}   | {improvement_str:10} | {winner_symbol}")
    else:
        status = "No Local Model" if not results['city_available'] else "Test Failed"
        print(f"{city:10} | {results['global_mae']:8.2f}   | {'---':8}   | {'---':10} | {status}")

# === SCENARIO-SPECIFIC ANALYSIS ===
if scenario_results:
    print(f"\n2. SPECIFIC SCENARIO ANALYSIS")
    print("   Testing challenging conditions where local expertise should excel")
    print()
    print(f"{'Scenario':<25} | {'Expected':<8} | {'Global':<7} | {'Local':<7} | {'Winner'}")
    print("-" * 70)
    
    for scenario_id, result in scenario_results.items():
        scenario_short = result['scenario'][:20] + "..." if len(result['scenario']) > 20 else result['scenario']
        winner_short = result['winner'].split('(')[0] if '(' in result['winner'] else result['winner']
        
        print(f"{scenario_short:<25} | {result['expected']:>8.0f} | {result['global_pred']:>7.0f} | "
              f"{result['city_pred']:>7.0f} | {winner_short}")

# === BUSINESS IMPACT SUMMARY ===
if total_improvements:
    avg_improvement = sum(total_improvements) / len(total_improvements)
    win_rate = (wins / total_comparisons) * 100
    
    print(f"\n3. BUSINESS IMPACT SUMMARY")
    print("-" * 30)
    print(f"Markets Successfully Compared: {total_comparisons}")
    print(f"Local Model Win Rate: {win_rate:.0f}%")
    print(f"Average Accuracy Improvement: {avg_improvement:+.1f}%")
    
    # Calculate business value
    if avg_improvement > 0:
        avg_error_reduction = sum([(comparison_results[city]['global_mae'] - comparison_results[city]['city_mae']) 
                                 for city in comparison_results.keys() 
                                 if comparison_results[city]['city_available'] and comparison_results[city]['winner'] == 'City'])
        avg_error_reduction = avg_error_reduction / wins if wins > 0 else 0
        print(f"Average Error Reduction: {avg_error_reduction:.1f} minutes per delivery")
        
        print(f"\nBUSINESS CASE ASSESSMENT:")
        if avg_improvement > 10:
            print("   STRONG BUSINESS CASE")
            print("   - Significant accuracy improvements across markets")
            print("   - Clear ROI from local optimization")
            print("   - Recommend full deployment of city-specific models")
        elif avg_improvement > 5:
            print("   MODERATE BUSINESS CASE") 
            print("   - Measurable improvements in key markets")
            print("   - ROI depends on operational costs")
            print("   - Recommend pilot deployment in top-performing cities")
        elif avg_improvement > 0:
            print("   EMERGING BUSINESS CASE")
            print("   - Small but consistent improvements")
            print("   - Framework shows promise")
            print("   - Recommend continued optimization")
        else:
            print("   INCONCLUSIVE")
            print("   - Mixed results require further analysis")
    
    # Scenario insights
    if scenario_results:
        scenario_wins = sum(1 for r in scenario_results.values() if r['improvement'] > 0)
        scenario_total = len(scenario_results)
        print(f"\nSCENARIO INSIGHTS:")
        print(f"   City models excel in {scenario_wins}/{scenario_total} challenging scenarios")
        print(f"   Local expertise most valuable during:")
        
        winning_scenarios = [r['scenario'] for r in scenario_results.values() if r['improvement'] > 0]
        for scenario in winning_scenarios:
            print(f"   - {scenario}")

# === TECHNICAL SUMMARY ===
print(f"\n4. TECHNICAL IMPLEMENTATION STATUS")
print("-" * 40)
print(f"Model Registry Status:")
print(f"   Global Model: delivery-time-global (Available)")
for city in city_datasets.keys():
    if city in successfully_registered:
        print(f"   {city} Model: delivery-time-{city.lower()} (Available)")
    else:
        print(f"   {city} Model: (Registration Failed)")

print(f"\nMLflow Integration:")
print(f"   Dashboard: http://127.0.0.1:8080")
print(f"   Model Registry: All models registered and versioned")
print(f"   Experiment Tracking: Training metrics and artifacts logged")
print(f"   Deployment Ready: Models can be deployed via MLflow serving")

print(f"\n5. FRAMEWORK BENEFITS DEMONSTRATED")
print("-" * 40)
print("Coffee Machine Approach Validated:")
print("   ✓ Same architecture deployed globally")
print("   ✓ Consistent preprocessing and model management")
print("   ✓ Local fine-tuning capabilities")
print("   ✓ Centralized model registry and versioning")
print("   ✓ Strategy-based prediction system")
print("   ✓ Production-ready deployment pipeline")

print(f"\n6. NEXT STEPS")
print("-" * 15)
print("For Production Deployment:")
print("   1. Implement automated model validation pipelines")
print("   2. Set up CI/CD for model deployment and updates")
print("   3. Add real-time model performance monitoring") 
print("   4. Implement A/B testing framework")
print("   5. Create feature stores for consistent data processing")
print("   6. Establish model retraining schedules")

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)
print("The Global Model Framework demonstrates measurable improvements")
print("when city-specific optimization is applied to delivery time prediction.")
print("Local expertise provides clear business value in challenging scenarios.")
print("=" * 70)

STEP 8: COMPREHENSIVE BUSINESS CASE REPORT
FOOD DELIVERY TIME PREDICTION - GLOBAL vs CITY-SPECIFIC MODELS

EXECUTIVE SUMMARY:
This analysis compares a single global model against city-specific models
to determine if local optimization provides measurable business value.

1. AGGREGATE PERFORMANCE (MAE = Mean Absolute Error)
   Lower MAE = More Accurate Predictions = Better Customer Experience

Market     | Global MAE | Local MAE  | Improvement | Winner
----------------------------------------------------------
NYC        |     9.65   |     9.30   |   +3.6%    | Local
Phoenix    |    10.91   |    10.74   |   +1.6%    | Local
London     |     9.75   |     9.48   |   +2.7%    | Local
Tokyo      |     9.19   |     8.49   |   +7.7%    | Local
Mumbai     |    15.92   |    15.68   |   +1.5%    | Local

2. SPECIFIC SCENARIO ANALYSIS
   Testing challenging conditions where local expertise should excel

Scenario                  | Expected | Global  | Local   | Winner
----------------------------